<b><font size="6" color="#E8800A">Week 13b: Choosing neural network settings for prices</font></b><br>
<b><font size="4">How the network is fitted</font></b><br>

This class keeps the layers and activations of the Week 12 joint network and changes how it is fitted: the solver, the L2 penalty, the learning rate and its schedule, and the stopping rule. A random search then combines these settings with the Week 12 choices.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>Optimisation changes the fitted network</font>](#optimisation)
* [<font color='#E8800A'>How backpropagation computes the gradient</font>](#backprop)
* [<font color='#E8800A'>Assess the unchanged reference</font>](#reference)
* [<font color='#E8800A'>MLP settings</font>](#settings)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The final search and its winner</font>](#search)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

In [ ]:
import sys
import joblib
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import time
import warnings
from functools import partial
from joblib import parallel_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, GridSearchCV, ParameterGrid, cross_validate, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


__Step 1:__ Import the course preprocessing tools.

In [ ]:
from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
from course_helpers import result_context
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"

__Step 2:__ Set the seed, configure the plots and filter the expected warnings.

In [ ]:
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, result_context, results_before

random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", None)
set_matplotlib_formats("svg")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (8, 5)})

ignore_unknown_category_warning = partial(
    warnings.filterwarnings, "ignore",
    message=r"^Found unknown categories in columns \[.*\] during transform\. These unknown categories will be encoded as all zeros$",
    category=UserWarning, module=r"^sklearn\.preprocessing\._encoders$")
ignore_unknown_category_warning()
def search_warning_policy():
    ignore_unknown_category_warning()
    warnings.filterwarnings("ignore",
        message=r"^Stochastic Optimizer: Maximum iterations \((200|300|3000)\) reached and the optimization hasn't converged yet\.$",
        category=ConvergenceWarning, module=r"^sklearn\.neural_network\._multilayer_perceptron$")
    warnings.filterwarnings("ignore",
        message=r"^lbfgs failed to converge after (?P<mlp_cap>200|300|3000) iteration\(s\) \(status=1\):\nSTOP: TOTAL NO\. OF ITERATIONS REACHED LIMIT\n\nIncrease the number of iterations to improve the convergence \(max_iter=(?P=mlp_cap)\)\.\nYou might also want to scale the data as shown in:\n    https://scikit-learn\.org/stable/modules/preprocessing\.html$",
        category=ConvergenceWarning, module=r"^sklearn\.neural_network\._multilayer_perceptron$")

__Step 3:__ Load the cars, set aside the test rows, and prepare the training folds.

In [ ]:
with parallel_config(backend='loky', initializer=ignore_unknown_category_warning):
    cars, regression_recipe = load_recipe('../../data/interim/cars4you.csv', regression_recipe_log)
    inherited_pipeline = joblib.load('../../results/week_12_regression_model.joblib')
    regression_recipe = inherited_pipeline.named_steps['preparation'].recipe
    X = cars.loc[:, regression_recipe['categorical'] + regression_recipe['numeric']].copy()
    y = cars['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
    cars_prep = FoldPreprocessor(regression_recipe).fit(X_train, y_train)
    X_train_ready = cars_prep.transform(X_train)
    X_test_ready = cars_prep.transform(X_test)
    cars_folds = outer_splitter('regression', random_state=random_state)
    cars_prepared = prepared_folds(FoldPreprocessor(regression_recipe), cars_folds, X_train, y_train, n_jobs=2)
    fixed_settings = inherited_pipeline.named_steps['model'].regressor.get_params(deep=False)
    default_network = clone(inherited_pipeline.named_steps['model'])
    cars_log = []
    print(f'{len(y_train):,} training cars and {len(y_test):,} test cars')

### The Week 12 winner

This is the fitted network saved at the end of Week 12. Its weights, layer sizes and activations provide the starting reference for the optimisation comparisons.

__Step 4:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = inherited_pipeline.named_steps['model'].regressor_
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = r"\hat y_{\mathrm{std}}"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 5:__ Tabulate the connections between the units drawn.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]

__Step 6:__ Label each layer with its fitted activation and array shapes.

In [ ]:
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + "\n" + r"$\hat y=\mu_y+\sigma_y\hat y_{\mathrm{std}}$ in euros"])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 7:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The Week 12 winner")
net_ax.axis("off")
plt.tight_layout()

__Step 8:__ Show the figure.

In [ ]:
plt.show()

# <font color='#E8800A'>Optimisation changes the fitted network</font> <a class="anchor" id="optimisation"></a>
[Back to TOC](#toc)

The joint network chosen in Week 12 and its complete preparation, target standardisation included, remain the reference here. The forward calculation stays the same; the comparisons change how the network is fitted.

The default training objective is half squared error plus an L2 penalty, not MAE:
$$L(\theta)=\frac{1}{2n}\sum_{i=1}^{n}(z_i-f_\theta(x_i))^2+\frac{\alpha}{2n}\sum_\ell\lVert W_\ell\rVert_F^2.$$
Here $n$ counts rows in this loss calculation: a minibatch for stochastic updates, or the full fitting partition for L-BFGS. The $x_i$ are those rows' preprocessed features and $z_i$ their prices standardised with the target mean and standard deviation of that training fit. The function $f_\theta$ predicts in those standardised units. The learned parameters $\theta$ contain weights and biases; $W_\ell$ is the weight matrix for layer $\ell$, and its squared Frobenius norm sums its squared entries. Biases are not penalised. Predictions are transformed back to euros before MAE is calculated. A smaller training objective therefore need not give the smallest validation MAE.

`solver` chooses the optimiser: Adam ([Kingma and Ba, 2015](https://arxiv.org/abs/1412.6980)) uses adaptive stochastic updates, SGD ([Robbins and Monro, 1951](https://doi.org/10.1214/aoms/1177729586)) uses stochastic gradient updates, and L-BFGS ([Liu and Nocedal, 1989](https://doi.org/10.1007/BF01589116)) uses a quasi-Newton method. L-BFGS does not use the stochastic learning-rate or early-stopping controls.

Prediction loss is half mean squared error, $\frac{1}{2n}\sum_i(\hat y_i^{(z)}-y_i^{(z)})^2$. Here $y_i^{(z)}$ is the price of row $i$ of $n$, standardised with the scaler fitted on the fitting partition, and $\hat y_i^{(z)}$ its prediction, so both losses share standardised units; MAE stays in euros.

Short iteration allowances leave optimisation unfinished. The diagnostic table counts the iteration-limit warnings of each fit below, the full-training fits included.

In [ ]:
fit_diagnostics = []
def note_fit(estimator, caught, label):
    expected = []
    for item in caught:
        known = (issubclass(item.category, ConvergenceWarning)
                 and ("Maximum iterations" in str(item.message)
                      or "TOTAL NO. OF ITERATIONS REACHED LIMIT" in str(item.message)))
        if known:
            expected.append(item)
        else:
            warnings.warn_explicit(item.message, item.category, item.filename, item.lineno)
    network = estimator.regressor_ if hasattr(estimator, "regressor_") else estimator
    fit_diagnostics.append({"fit": label, "iterations": network.n_iter_,
                            "iteration allowance": network.max_iter,
                            "convergence warnings": len(expected),
                            "status": ("iteration limit" if expected else "other convergence warning" if any(issubclass(w.category, ConvergenceWarning) for w in caught) else "stopping rule satisfied")})

In [ ]:
def prediction_diagnostics(estimator, fold):
    losses, scores = [], []
    for partition in ["fit", "score"]:
        features, target = fold[f"X_{partition}"], fold[f"y_{partition}"]
        target_z = estimator.transformer_.transform(np.asarray(target).reshape(-1, 1)).ravel()
        prediction_z = estimator.regressor_.predict(features)
        losses.append(0.5 * np.mean((prediction_z - target_z) ** 2))
        scores.append(mean_absolute_error(target, estimator.predict(features)))
    return [*losses, *scores]

### Follow each solver iteration

A single input and no hidden layer leave one weight and one bias to learn. The left panel shows their position on the loss surface; the right panel shows training and validation half squared error at that position. Each frame is a deterministic fit with the iteration allowance shown, using seed 33. The toy uses 24 training points and 18 separate validation points; solid lines show training loss and dashed lines validation loss.

SGD and Adam count passes through a full batch in this example. L-BFGS counts quasi-Newton iterations and can evaluate several trial steps per iteration. Equal iteration counts therefore represent different amounts of work. After a solver stops, its dot stays at the final position and its status says stopped.

![Solver positions and prediction loss at each iteration](../../assets/week_13/solver_regression.gif)

# <font color='#E8800A'>How backpropagation computes the gradient</font> <a class="anchor" id="backprop"></a>
[Back to TOC](#toc)

Backpropagation applies the chain rule to the training loss. This example has two inputs, two ReLU hidden units and one output: four input weights, two hidden biases, two output weights and one output bias. The following cells evaluate each stage using the supplied arrays.

Rows of $W_1$ index inputs and columns index hidden units. With one input row, the hidden weighted sums are $z=xW_1+b_1$, their activations are $h=\max(0,z)$, and the output weighted sum is $a=hW_2+b_2$. The infographic states the array shapes and distinguishes matrix multiplication from the elementwise ReLU derivative.

Regression predicts $\hat y=a$ and uses $L=\frac12(\hat y-y)^2$. Its output derivative is $\partial L/\partial a=\hat y-y$. The toy target is $y=1$ and the learning rate is $\eta=0.1$; these are arbitrary toy units, without an L2 penalty. Every gradient uses the parameters before the update. Apply all updates simultaneously, then recompute the prediction and loss.

### Follow the calculation through the graph

Each stage places the formula above its numerical substitution. Forward arrows carry values; backward arrows carry derivatives. The fixed example uses the arrays in the calculations below. Its final frames apply all parameter updates simultaneously and recompute the loss.

Matrix multiplication, written $@$ in NumPy, sums over the shared dimension. Here $xW_1$ has shape $(1,2)@(2,2)\to(1,2)$. The Hadamard product, $\odot$ or NumPy $*$, multiplies corresponding entries without summing. Backpropagation first computes $\delta_o W_2^T:(1,1)@(1,2)\to(1,2)$, then applies the ReLU derivative elementwise: $\delta_h=(\delta_o W_2^T)\odot\mathbf1[z>0]$. The weight gradient $x^T\delta_h$ is an outer matrix product, $(2,1)@(1,2)\to(2,2)$. For two-dimensional arrays, `np.dot(A, B)` and `A @ B` perform matrix multiplication; neither is the Hadamard product. A vector dot product that reduced the hidden derivatives to one scalar would lose the separate gradient for each hidden unit.

The animation follows the operation-by-operation presentation in [Stanford CS224N, neural networks and backpropagation](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture03-neuralnets.pdf). Its numbers use this notebook's two-unit example. Blue plus FORWARD denotes value computation; red plus BACKWARD denotes differentiation.

![Computational graph with formulas above numerical substitutions](../../assets/week_13/backprop_regression.gif)

### Parameters and shapes

The two columns of W1 belong to hidden units 1 and 2. There are four input weights, two hidden biases, two output weights and one output bias: nine scalar parameters.

In [ ]:
bp_x = np.array([[1.0, 2.0]])
bp_W1 = np.array([[0.1, -0.2], [0.2, 0.2]])
bp_b1 = np.array([0.1, 0.0])
bp_W2 = np.array([[0.5], [0.5]])
bp_b2 = np.array([0.0])
bp_y = 1.0
bp_eta = 0.1
print("Input and parameter shapes:", bp_x.shape, bp_W1.shape, bp_b1.shape, bp_W2.shape, bp_b2.shape)

### Hidden activations

$z_1=1(0.1)+2(0.2)+0.1=0.6$ and $z_2=1(-0.2)+2(0.2)+0=0.2$. Both are positive, so $h=[0.6,0.2]$. Both sums are away from zero, the point where ReLU has no derivative.

In [ ]:
bp_z = bp_x @ bp_W1 + bp_b1
bp_h = np.maximum(0.0, bp_z)
print("Hidden sums z:", bp_z)
print("ReLU activations h:", bp_h)

### Prediction and loss

$a=0.6(0.5)+0.2(0.5)+0=0.4$. Identity gives $\hat y=a=0.4$, so $L=\frac12(a-y)^2=0.18$. There is no averaging or penalty term.

In [ ]:
bp_a = bp_h @ bp_W2 + bp_b2
bp_prediction = bp_a.copy()
bp_loss = (0.5 * (bp_prediction-bp_y)**2).item()
print("Output weighted sum and identity prediction:", bp_prediction)
print("Half squared error:", bp_loss)

### Output derivative

Because identity has derivative 1, $\delta_{out}=\partial L/\partial a=a-y=-0.6$. The prediction is 0.4 and the signed derivative is -0.6. Keep this signal as a $(1,1)$ array.

In [ ]:
bp_delta_out = bp_a-bp_y
print("Output loss signal:", bp_delta_out, "shape:", bp_delta_out.shape)

### Output-weight gradients

The gradients $dW_2=\partial L/\partial W_2$ and $db_2=\partial L/\partial b_2$ are arrays with the same shape as the parameter they belong to, so $(2,1)$ and $(1,)$. The transpose $h^T$ turns the $(1,2)$ row $h$ into a $(2,1)$ column, and `.T` is the NumPy spelling, so $dW_2=h^T\delta_{out}$ has shape $(2,1)$. The bias derivative is $\delta_{out}$ summed over its single row, which is what `.sum(axis=0)` does, and it has shape $(1,)$. The two weight derivatives are $0.6(-0.6)=-0.36$ and $0.2(-0.6)=-0.12$; the bias derivative is $-0.6$.

In [ ]:
bp_dW2 = bp_h.T @ bp_delta_out
bp_db2 = bp_delta_out.sum(axis=0)
print("dW2:", bp_dW2, "db2:", bp_db2)

### Hidden derivatives

$\delta_{hidden}=(\delta_{out}W_2^T)\odot\mathbf{1}(z>0)$, where $\odot$ means elementwise multiplication and $\mathbf{1}(z>0)$ is the ReLU derivative. For unit 1, $-0.6(0.5)(1)=-0.3$. Unit 2 receives the same signal. A negative hidden sum would give a zero ReLU derivative and block the derivative through that unit. Neither sum here is negative.

In [ ]:
bp_delta_hidden = (bp_delta_out @ bp_W2.T) * (bp_z > 0)
print("Hidden loss signals:", bp_delta_hidden)

### Input-weight gradients

Likewise $dW_1=\partial L/\partial W_1$ and $db_1=\partial L/\partial b_1$ have the shapes of $W_1$ and $b_1$. Here $x^T$ is a $(2,1)$ column, so $dW_1=x^T\delta_{hidden}$ has shape $(2,2)$, and summing the hidden signal over its single row gives $db_1$ with shape $(2,)$. The rows of $dW_1$ are $[-0.3,-0.3]$ and $[-0.6,-0.6]$ because the inputs are 1 and 2. Both bias derivatives are $-0.3$.

In [ ]:
bp_dW1 = bp_x.T @ bp_delta_hidden
bp_db1 = bp_delta_hidden.sum(axis=0)
print("dW1:", bp_dW1)
print("db1:", bp_db1)

### Parameter update

Each parameter after the update equals its value before the update minus $\eta$ times its derivative, the plain gradient-descent rule behind the `sgd` solver. Compute all four gradient arrays before changing any parameter. $W_1^{new}=[[0.13,-0.17],[0.26,0.26]]$, $b_1^{new}=[0.13,0.03]$, $W_2^{new}=[[0.536],[0.512]]$, and $b_2^{new}=[0.06]$. A scikit-learn `sgd` update also averages the gradient over a minibatch of rows and adds momentum by default.

In [ ]:
bp_W1_new = bp_W1-bp_eta*bp_dW1
bp_b1_new = bp_b1-bp_eta*bp_db1
bp_W2_new = bp_W2-bp_eta*bp_dW2
bp_b2_new = bp_b2-bp_eta*bp_db2
print("New W1:", bp_W1_new, "new b1:", bp_b1_new)
print("New W2:", bp_W2_new, "new b2:", bp_b2_new)

### Prediction after the update

After the update, the hidden sums are $[0.78,0.38]$. The identity prediction becomes $0.67264$, and loss falls from $0.18$ to $0.0535822848$.

In [ ]:
bp_z_new = bp_x @ bp_W1_new+bp_b1_new
bp_h_new = np.maximum(0.0, bp_z_new)
bp_a_new = bp_h_new @ bp_W2_new+bp_b2_new
bp_prediction_new = bp_a_new.copy()
bp_loss_new = (0.5*(bp_prediction_new-bp_y)**2).item()
print("New hidden sums:", bp_z_new)
print("New output weighted sum:", bp_a_new, "new prediction:", bp_prediction_new)
print("Loss before:", bp_loss, "loss after:", bp_loss_new)
assert bp_loss_new < bp_loss

### Using the same parameter values

All derivatives must describe the same loss at the same parameter values. Updating an output weight early changes the chain-rule multiplier for the hidden layer and mixes two parameter states. Storing the updated arrays separately preserves the parameters from before the update until every derivative has been computed.

# <font color='#E8800A'>Assess the unchanged reference</font> <a class="anchor" id="reference"></a>
[Back to TOC](#toc)

Ten repeated course folds score the Week 12 joint network, target standardisation included, before any setting changes.

__Step 9:__ Score the prepared reference on the ten prepared folds.

In [ ]:
reference_train, reference_validation = ([], [])
for fold in tqdm(cars_prepared, desc='Course-fold assessment', unit='fold', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        fitted_fold = clone(default_network).fit(fold['X_fit'], fold['y_fit'])
    note_fit(fitted_fold, caught, 'reference, fold')
    reference_train.append(mean_absolute_error(fold['y_fit'], fitted_fold.predict(fold['X_fit'])))
    reference_validation.append(mean_absolute_error(fold['y_score'], fitted_fold.predict(fold['X_score'])))
print('Reference validation MAE by fold (EUR):', np.round(reference_validation, 2))

__Step 10:__ Fit the reference on all training cars.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    fitted_reference = clone(default_network).fit(X_train_ready, y_train)
note_fit(fitted_reference, caught, 'reference, all training rows')

__Step 11:__ Log the unchanged reference with its training, validation and held-out MAE.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": str(default_network),
    "train": reference_train, "validation": reference_validation,
    "model": fitted_reference}, X_test_ready, y_test,
    "selected in Week 12", metric=mean_absolute_error, label="MAE")

**The Week 12 joint reference has validation MAE 2,565.90 EUR and test MAE 2,582.21 EUR, with mean training MAE 2,297.45 EUR.** It has one ReLU layer of 100 units, batch size 200 and a 200-iteration allowance.

Each checkpoint refits the network on the first prepared training fold with the iteration allowance on the horizontal axis, then scores that fold's training and validation rows. The losses omit the L2 penalty, which the optimiser also minimises.

__Step 12:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
curve_values = ['reference']
curve_base = clone(default_network)

__Step 13:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"regressor__max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"Reference network: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit.regressor_
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 14:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="half MSE (standardised price)", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="MAE (EUR)", title="Task score: training and validation")
curve_fig.suptitle('Reference network')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

# <font color='#E8800A'>MLP settings</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Each sweep changes one argument of the Week 12 reference on the same ten training folds, and the row with the lowest mean validation MAE is logged. The SGD-schedule and validation-fraction sweeps also set the solver or early stopping they need. Every tested value enters the joint search.

| Parameter | Meaning |
| --- | --- |
| `solver` | Optimisation algorithm: Adam, SGD or L-BFGS. |
| `alpha` | Strength of the L2 penalty on the weights. |
| `learning_rate_init` | Initial step size of the stochastic solvers. |
| `learning_rate` | Schedule of the step size under SGD. |
| `early_stopping` | Whether fitting stops when the score on an internal validation subset fails to improve. |
| `validation_fraction` | Share of the fitting rows held back for that internal check. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)

### `MLP: solver`

The solvers update the same network through different optimisation rules: Adam uses adaptive stochastic updates, SGD uses stochastic gradient updates, and L-BFGS uses a quasi-Newton method on the whole training matrix. The candidate list is `["adam", "sgd", "lbfgs"]`.

__Step 15:__ Set the `solver` values.

In [ ]:
mlp_solver_values = ["adam", "sgd", "lbfgs"]

In [ ]:
# The values of this one setting were set above; every other setting stays at the reference.
# One row per value will hold its scores, its seconds and its fitted model.
mlp_solver_rows = []

__Step 16:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_solver_values, desc='solver values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__solver=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'solver={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'solver={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_solver_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 17:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_solver_sweep = pd.DataFrame(mlp_solver_rows)
display(mlp_solver_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_solver_winner = mlp_solver_sweep.loc[mlp_solver_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_solver_winner["configuration"], "train": mlp_solver_winner["train"],
    "validation": mlp_solver_winner["validation"], "model": mlp_solver_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 18:__ Plot the training and validation MAE for `solver`.

In [ ]:
# Training and validation MAE for each solver value.
parameter_positions = np.arange(len(mlp_solver_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_solver_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_solver_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="solver", xticks=parameter_positions, xticklabels=mlp_solver_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Adam gives the lowest validation MAE, 2,565.90 EUR, followed by L-BFGS at 2,617.63 and SGD at 3,365.93.** L-BFGS fits the training folds most closely, at MAE 2,188.70 EUR, and has the lowest test MAE, 2,445.00 EUR. The row with the lowest validation MAE is logged. All three full fits use 200 iterations and all three raise warnings in ten folds.

__Step 19:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
curve_values = mlp_solver_values
curve_base = clone(default_network)

__Step 20:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"regressor__solver": curve_value, "regressor__max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"solver: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit.regressor_
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 21:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="half MSE (standardised price)", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="MAE (EUR)", title="Task score: training and validation")
curve_fig.suptitle('solver')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: alpha`

`alpha` controls the L2 penalty on large weights. A larger penalty can reduce sensitivity to training details, but can also prevent useful relationships from being fitted. The candidate list is `[1e-4, 1e-2, 1.0]`.

__Step 22:__ Set the `alpha` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_alpha_values = [1e-4, 1e-2, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_alpha_rows = []

__Step 23:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_alpha_values, desc='alpha values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__alpha=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'alpha={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'alpha={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_alpha_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 24:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_alpha_sweep = pd.DataFrame(mlp_alpha_rows)
display(mlp_alpha_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_alpha_winner = mlp_alpha_sweep.loc[mlp_alpha_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_alpha_winner["configuration"], "train": mlp_alpha_winner["train"],
    "validation": mlp_alpha_winner["validation"], "model": mlp_alpha_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 25:__ Plot the training and validation MAE for `alpha`.

In [ ]:
# Training and validation MAE for each alpha value.
parameter_positions = np.arange(len(mlp_alpha_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_alpha_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_alpha_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="alpha", xticks=parameter_positions, xticklabels=mlp_alpha_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**The reference penalty, `alpha=0.0001`, has validation MAE 2,565.90 EUR, below 2,570.88 at 0.01 and 2,639.20 at 1.0.** The largest penalty raises training MAE from 2,297.45 to 2,555.73 EUR and stops the full fit after 167 iterations. At 0.01, test MAE is slightly lower than the reference, 2,579.27 against 2,582.21 EUR, while validation MAE is higher.

### `MLP: learning_rate_init`

`learning_rate_init` sets the starting step size of Adam and SGD; this sweep uses Adam.

A larger rate takes larger steps, which may reduce loss faster or overshoot a minimum. The candidate list is `[1e-3, 3e-3, 1e-2]`.

__Step 26:__ Set the `learning_rate_init` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_learning_rate_init_values = [1e-3, 3e-3, 1e-2]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_learning_rate_init_rows = []

__Step 27:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_learning_rate_init_values, desc='learning_rate_init values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__learning_rate_init=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'learning_rate_init={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'learning_rate_init={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_learning_rate_init_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 28:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_learning_rate_init_sweep = pd.DataFrame(mlp_learning_rate_init_rows)
display(mlp_learning_rate_init_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_learning_rate_init_winner = mlp_learning_rate_init_sweep.loc[mlp_learning_rate_init_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_learning_rate_init_winner["configuration"], "train": mlp_learning_rate_init_winner["train"],
    "validation": mlp_learning_rate_init_winner["validation"], "model": mlp_learning_rate_init_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 29:__ Plot the training and validation MAE for `learning_rate_init`.

In [ ]:
# Training and validation MAE for each learning_rate_init value.
parameter_positions = np.arange(len(mlp_learning_rate_init_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_learning_rate_init_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_learning_rate_init_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="learning_rate_init", xticks=parameter_positions, xticklabels=mlp_learning_rate_init_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Validation MAE is lowest at rate 0.001, 2,565.90 EUR, and rises to 2,620.27 at 0.003 and 2,690.11 at 0.01.** The two larger rates stop the full fit after 120 and 115 iterations, against the full allowance of 200 at 0.001. Their test MAEs are lower, at 2,563.75 and 2,563.12 EUR, so the validation and test rankings differ.

__Step 30:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
curve_values = mlp_learning_rate_init_values
curve_base = clone(default_network)

__Step 31:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"regressor__learning_rate_init": curve_value, "regressor__max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"learning_rate_init: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit.regressor_
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 32:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="half MSE (standardised price)", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="MAE (EUR)", title="Task score: training and validation")
curve_fig.suptitle('learning_rate_init')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: learning_rate`

For SGD, `learning_rate` sets the schedule: `constant` holds the step size fixed, `adaptive` divides it by five when progress stalls, and `invscaling` follows
$$\eta_t=\eta_0/(t+1)^p.$$
Here $\eta_t$ is the step size after $t$ training examples have been seen, $\eta_0$ is `learning_rate_init`, and $p$ is `power_t`, held at its default 0.5. A shrinking step can stop substantial learning before the network predicts well, so a fit can satisfy its stopping rule while its validation MAE remains high. Every schedule row uses `solver='sgd'`, the only solver that reads this setting. The candidate list is `["constant", "adaptive", "invscaling"]`.

__Step 33:__ Set the `learning_rate` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_learning_rate_values = ["constant", "adaptive", "invscaling"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_learning_rate_rows = []

__Step 34:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_learning_rate_values, desc='learning_rate values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__solver='sgd', regressor__learning_rate=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'learning_rate={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'learning_rate={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_learning_rate_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 35:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_learning_rate_sweep = pd.DataFrame(mlp_learning_rate_rows)
display(mlp_learning_rate_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_learning_rate_winner = mlp_learning_rate_sweep.loc[mlp_learning_rate_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_learning_rate_winner["configuration"], "train": mlp_learning_rate_winner["train"],
    "validation": mlp_learning_rate_winner["validation"], "model": mlp_learning_rate_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 36:__ Plot the training and validation MAE for `learning_rate`.

In [ ]:
# Training and validation MAE for each learning_rate value.
parameter_positions = np.arange(len(mlp_learning_rate_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_learning_rate_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_learning_rate_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="learning_rate", xticks=parameter_positions, xticklabels=mlp_learning_rate_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Constant and adaptive SGD both give validation MAE 3,365.93 EUR at the 200-iteration allowance; inverse scaling gives 6,837.56 EUR.** The first two raise warnings on all ten folds. Inverse scaling stops after 39 iterations without a fold warning, while training MAE remains 6,838.95 EUR.

__Step 37:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
curve_values = mlp_learning_rate_values
curve_base = clone(default_network)

__Step 38:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"regressor__learning_rate": curve_value, "regressor__solver": "sgd", "regressor__max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"learning_rate: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit.regressor_
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 39:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="half MSE (standardised price)", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="MAE (EUR)", title="Task score: training and validation")
curve_fig.suptitle('learning_rate')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: early_stopping`

`early_stopping=True` reserves part of each fitting partition to monitor validation R² and stops when that score fails to improve for the default patience of 10 epochs. Feature preprocessing and target standardisation have already seen this internal subset, unlike the external validation fold. With early stopping disabled, the stopping rule monitors training loss. The candidate list is `[False, True]`.

__Step 40:__ Set the `early_stopping` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_early_stopping_values = [False, True]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_early_stopping_rows = []

__Step 41:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_early_stopping_values, desc='early_stopping values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__early_stopping=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'early_stopping={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'early_stopping={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_early_stopping_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 42:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_early_stopping_sweep = pd.DataFrame(mlp_early_stopping_rows)
display(mlp_early_stopping_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_early_stopping_winner = mlp_early_stopping_sweep.loc[mlp_early_stopping_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_early_stopping_winner["configuration"], "train": mlp_early_stopping_winner["train"],
    "validation": mlp_early_stopping_winner["validation"], "model": mlp_early_stopping_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 43:__ Plot the training and validation MAE for `early_stopping`.

In [ ]:
# Training and validation MAE for each early_stopping value.
parameter_positions = np.arange(len(mlp_early_stopping_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_early_stopping_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_early_stopping_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="early_stopping", xticks=parameter_positions, xticklabels=mlp_early_stopping_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**With early stopping enabled, validation MAE rises from 2,565.90 to 2,640.63 EUR.** Both full fits use all 200 allowed iterations; the early-stopping fit raises a warning on one fold, against ten without it. Its test MAE is 2,583.98 EUR, close to the reference 2,582.21 EUR.

__Step 44:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
curve_values = [False, True]
curve_base = clone(default_network)

__Step 45:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"regressor__early_stopping": curve_value, "regressor__max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"early_stopping: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit.regressor_
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 46:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="half MSE (standardised price)", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="MAE (EUR)", title="Task score: training and validation")
curve_fig.suptitle('early_stopping')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

Early stopping watches R² on its internal subset, while the right panel shows MAE on the external fold. The left panel scores the restored weights with the same loss on the fold's training and validation rows.

### `MLP: validation_fraction`

`validation_fraction` controls the reserved proportion. Both fractions are tested with early stopping enabled; the fraction has a role only for the stochastic solvers.

`0.1` reserves 10 percent of each fitting partition for the stopping check and leaves 90 percent for weight updates; `0.2` reserves 20 percent and leaves 80 percent. A larger check has more observations but trains the weights on fewer rows. The candidate list is `[0.1, 0.2]`.

__Step 47:__ Set the `validation_fraction` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_validation_fraction_values = [0.1, 0.2]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_validation_fraction_rows = []

__Step 48:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_validation_fraction_values, desc='validation_fraction values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__early_stopping=True, regressor__validation_fraction=value)
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = clone(candidate).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'validation_fraction={value}, fold')
        train_scores.append(mean_absolute_error(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(mean_absolute_error(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    note_fit(deployed, caught, f'validation_fraction={value}, all training rows')
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_validation_fraction_rows.append({'configuration': 'TransformedTargetRegressor(regressor=MLPRegressor(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + '), transformer=StandardScaler())', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_MAE': np.mean(train_scores), 'cv_MAE': np.mean(validation_scores), 'test_MAE': mean_absolute_error(y_test, deployed.predict(X_test_ready)), 'iterations': deployed.regressor_.n_iter_, 'loss': deployed.regressor_.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': candidate.get_params(deep=False), 'model': deployed})

__Step 49:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_validation_fraction_sweep = pd.DataFrame(mlp_validation_fraction_rows)
display(mlp_validation_fraction_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_validation_fraction_winner = mlp_validation_fraction_sweep.loc[mlp_validation_fraction_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": len(cars_prepared),
    "name": mlp_validation_fraction_winner["configuration"], "train": mlp_validation_fraction_winner["train"],
    "validation": mlp_validation_fraction_winner["validation"], "model": mlp_validation_fraction_winner["model"]},
    X_test_ready, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

__Step 50:__ Plot the training and validation MAE for `validation_fraction`.

In [ ]:
# Training and validation MAE for each validation_fraction value.
parameter_positions = np.arange(len(mlp_validation_fraction_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_validation_fraction_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_validation_fraction_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="validation_fraction", xticks=parameter_positions, xticklabels=mlp_validation_fraction_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**With early stopping enabled, a fraction of 0.2 has validation MAE 2,628.66 EUR, below 2,640.63 at 0.1.** Test MAE moves in the other direction, from 2,583.98 to 2,619.76 EUR. Both full fits reach 200 iterations, and each setting raises one fold convergence warning.

**Initialisation also changes a fitted network.** The comparison changes only the reference network's seed, on the fixed training and test split.

__Step 51:__ Measure a bounded seed comparison on the fixed split.

In [ ]:
seed_rows = [{'seed': random_state, 'test MAE': mean_absolute_error(y_test, fitted_reference.predict(X_test_ready))}]
for seed in tqdm([34, 35], desc='Initialisation comparison', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model = clone(default_network).set_params(regressor__random_state=seed).fit(X_train_ready, y_train)
    note_fit(model, caught, f'seed {seed}, all training rows')
    seed_rows.append({'seed': seed, 'test MAE': mean_absolute_error(y_test, model.predict(X_test_ready))})
seed_table = pd.DataFrame(seed_rows)
display(seed_table)
print(f"Observed seed spread: {seed_table['test MAE'].max() - seed_table['test MAE'].min():,.2f} EUR")

**Seeds 33, 34 and 35 give test MAE 2,582.21, 2,535.90 and 2,569.44 EUR, a spread of 46.31 EUR.** These held-out scores describe initialisation variability.

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

Nested cross-validation scores the joint search on the ten outer folds before the final search is fitted.

A full Cartesian search would fit every valid combination. The finite domain combines the activation functions, hidden-layer sizes and iteration budgets studied in Week 12 with every optimisation value compared here. Conditional subgrids exclude settings a solver does not use, and include `validation_fraction` only when early stopping is enabled.

**The budget is 60 unique configurations**, drawn uniformly from the full `ParameterGrid`. Large branches receive more draws in expectation because they contain more configurations; a small branch can receive none.

[Bergstra and Bengio (2012)](https://jmlr.org/papers/v13/bergstra12a.html) motivate random search when exhaustive combinations are expensive. Scikit-learn's [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) samples a finite list the same way, without replacement. Drawing the list once gives every outer search and the final search the same candidates, and lets `CachedSearchCV` reuse fold preprocessing.

__Step 52:__ Define the conditional domain and draw 60 of its combinations without replacement.

In [ ]:
mlp_activation_values = ["identity", "logistic", "tanh", "relu"]
mlp_hidden_layer_sizes_values = [(8,), (16,), (32,), (64,), (100,), (32, 32), (64, 32), (32, 32, 32), (64, 32, 16), (64, 64, 64), (32, 64, 32), (32, 64)]
mlp_max_iter_values = [200, 300, 3000]
mlp_batch_size_values = [64, 200, 512]

In [ ]:
# The joint search combines the Week 12 choices with these optimisation settings.
common_grid = {
    "estimator__regressor__activation": mlp_activation_values,
    "estimator__regressor__hidden_layer_sizes": mlp_hidden_layer_sizes_values,
    "estimator__regressor__max_iter": mlp_max_iter_values,
    "estimator__regressor__alpha": mlp_alpha_values,
}
stochastic_grid = {
    "estimator__regressor__batch_size": mlp_batch_size_values,
    "estimator__regressor__learning_rate_init": mlp_learning_rate_init_values,
}
# L-BFGS has no stochastic settings; Adam and SGD each branch on early stopping, and the schedule applies to SGD only.
conditional_grid = [{**common_grid, "estimator__regressor__solver": [solver for solver in mlp_solver_values if solver == "lbfgs"]}]
for solver in [solver for solver in mlp_solver_values if solver != "lbfgs"]:
    solver_grid = {**common_grid, **stochastic_grid, "estimator__regressor__solver": [solver]}
    if solver == "sgd":
        solver_grid["estimator__regressor__learning_rate"] = mlp_learning_rate_values
    conditional_grid.append({**solver_grid, "estimator__regressor__early_stopping": [value for value in mlp_early_stopping_values if not value]})
    conditional_grid.append({**solver_grid, "estimator__regressor__early_stopping": [value for value in mlp_early_stopping_values if value],
                             "estimator__regressor__validation_fraction": mlp_validation_fraction_values})
mlp_param_grid = conditional_grid
full_space = ParameterGrid(mlp_param_grid)
candidate_indices = (np.random.RandomState(33).choice(len(full_space), size=60, replace=False)
                     if len(full_space) > 60 else np.arange(len(full_space)))

__Step 53:__ Assemble the sampled candidate configurations.

In [ ]:
mlp_candidates = [{key: [value] for key, value in full_space[int(index)].items()}
                  for index in candidate_indices]

__Step 54:__ Count the full domain and the sample by solver.

In [ ]:
full_solver_counts = pd.Series([configuration["estimator__regressor__solver"] for configuration in full_space]).value_counts()
sample_solver_counts = pd.Series([configuration["estimator__regressor__solver"][0] for configuration in mlp_candidates]).value_counts()
display(pd.DataFrame({"full domain": full_solver_counts, "sampled": sample_solver_counts}).fillna(0).astype(int))
print(f"Full domain: {len(full_space):,}; evaluated configurations: {len(mlp_candidates)}")
inner_folds = KFold(n_splits=3, shuffle=True, random_state=random_state)
base_mlp = PreparedEstimator(FoldPreprocessor(regression_recipe), default_network)
search_template = CachedSearchCV(GridSearchCV(base_mlp, mlp_candidates,
    scoring="neg_mean_absolute_error", cv=inner_folds, n_jobs=2, error_score="raise"))

**The conditional domain contains 47,088 configurations: 432 for L-BFGS, 11,664 for Adam and 34,992 for SGD.** The fixed draw contains 49 SGD, nine Adam and two L-BFGS configurations. Its sixty candidates cover 0.1274% of the domain.

Each outer training part runs the sixty-candidate search on three inner folds. The Week 12 joint reference uses the same ten outer folds, so the two are compared fold by fold.

The search silences the scikit-learn iteration-limit warning; the sweeps above show where it occurs. A low validation MAE can come from unfinished optimisation.

__Step 55:__ Run the bounded search inside every outer training fold.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    with parallel_config(backend="loky", initializer=search_warning_policy):
        with warnings.catch_warnings(record=True) as nested_caught:
            warnings.simplefilter("always", ConvergenceWarning)
            search_warning_policy()
            nested_fold_results = [cross_validate(search_template, X_train, y_train,
                scoring="neg_mean_absolute_error", cv=[outer_split],
                return_train_score=True, n_jobs=1, error_score="raise")
                for outer_split in tqdm(list(cars_folds.split(X_train, y_train)), total=cars_folds.get_n_splits(X_train, y_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
            nested_scores = {key: np.concatenate([result[key] for result in nested_fold_results])
                             for key in nested_fold_results[0]}
        nested_train = -nested_scores["train_score"]
        nested_validation = -nested_scores["test_score"]
        paired = pd.DataFrame({"reference MAE": reference_validation,
                               "search MAE": nested_validation})
        paired["improvement EUR"] = paired["reference MAE"] - paired["search MAE"]
        display(paired)
        print(f"Mean paired improvement: {paired['improvement EUR'].mean():+,.2f} EUR")
        print(f"Search wins: {(paired['improvement EUR'] > 0).sum()}/{len(paired)}")

In [ ]:
fig, ax = plt.subplots()
for row in paired.itertuples(index=False, name=None):
    ax.plot([0, 1], row[:2], color="0.75", linewidth=1)
ax.scatter(np.zeros(len(paired)), paired["reference MAE"], color=PLOT_BLUE, label="Week 12 joint reference")
ax.scatter(np.ones(len(paired)), paired["search MAE"], color=PLOT_ORANGE, label="Nested search")
ax.set(xticks=[0, 1], xticklabels=["Week 12 joint reference", "Nested search"], ylabel="Outer validation MAE (EUR)", title="Paired outer-fold comparison")
ax.legend()
fig.tight_layout()
plt.show()

**Nested search reaches mean MAE 2,548.09 EUR, against 2,565.90 EUR for the Week 12 joint reference.** Its mean paired reduction is 17.80 EUR, with lower error on six of ten folds and an outer-score standard deviation of 167.04 EUR. This is a modest difference, and the reference was itself selected on these folds, so its 2,565.90 EUR is optimistic.

# <font color='#E8800A'>The final search and its winner</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The final search runs the same sixty candidates and inner-fold rule on every training car and refits the configuration with the lowest mean inner MAE.

__Step 56:__ Select on all training cars and inspect the fitted winner.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    with parallel_config(backend='loky', initializer=search_warning_policy):
        with warnings.catch_warnings(record=True) as final_caught:
            warnings.simplefilter('always', ConvergenceWarning)
            search_warning_policy()
            search = clone(search_template).fit(X_train, y_train)

__Step 57:__ Read the five best candidates in inner-CV order.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    ranking = pd.DataFrame({"settings": search.cv_results_["params"],
        "inner MAE": -np.asarray(search.cv_results_["mean_test_score"])})
    top_five = ranking[np.isfinite(ranking["inner MAE"])].nsmallest(5, "inner MAE").copy()

__Step 58:__ Refit the five ranked candidates and report their held-out errors.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    top_five['test MAE'] = [mean_absolute_error(y_test, clone(search.search_.estimator).set_params(**settings).fit(X_train, y_train).predict(X_test)) for settings in tqdm(top_five['settings'], desc='top five candidates', leave=True, position=0)]
    display(top_five)

__Step 59:__ Read the selected settings and the complete winner constructor.

In [ ]:
chosen_settings = {key.removeprefix("estimator__regressor__"): value for key, value in search.best_params_.items()}
actual_settings = {**fixed_settings, **chosen_settings}
winner_name = "TransformedTargetRegressor(regressor=MLPRegressor(" + ", ".join(f"{key}={value!r}" for key, value in sorted(actual_settings.items())) + "), transformer=StandardScaler())"
print(winner_name)
print(f"Inner selection MAE: {-search.best_score_:,.2f} EUR")
print(f"Test MAE: {mean_absolute_error(y_test, search.predict(X_test)):,.2f} EUR")
winner_network = search.best_estimator_.estimator_

__Step 60:__ Score the winner on the shared folds.

In [ ]:
with warnings.catch_warnings(record=True) as winner_caught:
    warnings.simplefilter('always', ConvergenceWarning)
    winner_train, winner_validation = ([], [])
    for fold in tqdm(cars_prepared, desc='Course-fold assessment', unit='fold', leave=True, position=0):
        fitted_fold = clone(search.best_estimator_.estimator_).fit(fold['X_fit'], fold['y_fit'])
        winner_train.append(mean_absolute_error(fold['y_fit'], fitted_fold.predict(fold['X_fit'])))
        winner_validation.append(mean_absolute_error(fold['y_score'], fitted_fold.predict(fold['X_score'])))
note_fit(fitted_fold, winner_caught, 'winner, shared folds')
winner_warnings = sum((issubclass(w.category, ConvergenceWarning) for w in winner_caught))
print(f'Winner convergence warnings on the shared folds: {winner_warnings}')

__Step 61:__ Record the selected network with its own-fold estimate.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": winner_name,
    "train": winner_train, "validation": winner_validation,
    "model": search}, X_test, y_test,
    "winner's own score", metric=mean_absolute_error, label="MAE")

__Step 62:__ Record the nested estimate of the search.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": f"joint search over {len(mlp_candidates)} sampled combinations",
    "train": nested_train, "validation": nested_validation,
    "model": search}, X_test, y_test,
    "nested estimate", metric=mean_absolute_error, label="MAE")

**The final search selects three 64-unit ReLU layers with SGD, `alpha=1.0`, batch size 200 and initial learning rate 0.003.** The full fit stops after 890 of the 3,000 allowed iterations with early stopping off. Its inner selection MAE is 2,506.37 EUR and test MAE 2,550.66 EUR. Its own shared-fold MAE is 2,536.38 EUR, with no convergence warning; the board uses the nested estimate of 2,548.09 EUR.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The board reports nested MAE for the search procedure and test MAE for its final model, alongside scores from previous course weeks including the Week 12 joint network. The baseline predicts the training mean for every test car.

__Step 63:__ Save the nested search estimate and the fitted final network.

In [ ]:
results_dir = "../../results"
context = result_context("../../data/interim/cars4you.csv", regression_recipe_log,
    X_train.index, X_test.index, outer_folds=outer_folds(cars_folds, X_train, y_train))
floor = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
board = Scoreboard("Cars4You", "MAE (EUR)", floor=floor, context=context)
for earlier in results_before(13, "regression", results_dir, context=context):
    board.record(*earlier)
board.record(13, winner_name, float(np.mean(nested_validation)),
             mean_absolute_error(y_test, search.predict(X_test)))
print(board.table())
board.save(results_dir, 13, "regression",
    baseline_cv=float(np.mean(reference_validation)),
    baseline_test=mean_absolute_error(y_test, fitted_reference.predict(X_test_ready)))
chosen_model = search.best_estimator_
joblib.dump(chosen_model, f"{results_dir}/week_13_regression_model.joblib")

**The search enters the board at nested MAE 2,548.09 EUR and test MAE 2,550.66 EUR, against 2,565.90 and 2,582.21 EUR for the Week 12 joint reference.** The nested reduction is 17.80 EUR.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

- Optimisation, regularisation, capacity and stopping settings affect different parts of fitting. Applicable combinations depend on the solver.
- Backpropagation applies the chain rule from the loss back to every weight and bias, using the parameter values before the update.
- One-setting sweeps describe each setting's effect; the joint search combines every tested value.
- Random search draws sixty candidates from a domain too large to fit exhaustively.

### The Week 13 winner

The drawing uses the fitted network selected by the final training search. Layer sizes and activations come from its fitted arrays; ellipses mark units omitted from the drawing.

__Step 64:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = winner_network.regressor_
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = r"\hat y_{\mathrm{std}}"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 65:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]

__Step 66:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)

__Step 67:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + "\n" + r"$\hat y=\mu_y+\sigma_y\hat y_{\mathrm{std}}$ in euros"])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 68:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The final search winner")
net_ax.axis("off")
plt.tight_layout()

__Step 69:__ Show the figure.

In [ ]:
plt.show()

**The final winner has three hidden layers of 64 ReLU units and an identity output.** Its weight matrices are 21 by 64, 64 by 64, 64 by 64 and 64 by 1; the largest absolute weight is 0.771. The target transformer returns the identity output to euros before predictions are scored.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- [Bergstra and Bengio (2012), Random Search for Hyper-Parameter Optimization](https://jmlr.org/papers/v13/bergstra12a.html).
- [Robbins and Monro (1951), A Stochastic Approximation Method](https://doi.org/10.1214/aoms/1177729586).
- [Kingma and Ba (2015), Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980).
- [Liu and Nocedal (1989), On the limited memory BFGS method for large scale optimization](https://doi.org/10.1007/BF01589116).
- [Scikit-learn: RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).
- [Scikit-learn: MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).
- [Scikit-learn: nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).
- [Scikit-learn: TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).